# Preprocessing Prototype

**Project:** Power Consumption MLOps (Tetouan City)  
**Goal:** Prototype the preprocessing logic that will be extracted into `DataProcessor` in Phase 1.  
**Output:** Working preprocessing code in this notebook → extract to `src/tetouan_power/data_processor.py`.

> **Workflow:** Phase 0 (EDA in `00_initial_eda.ipynb`) → This notebook (prototype) → Phase 1 (extract to `src/`).  
> Do not write `DataProcessor` from scratch in `src/`. First validate the logic here.

## 1) Setup & Load Config

In [1]:
from pathlib import Path
import pandas as pd

from tetouan_power.config import ProjectConfig

# Paths (same as EDA notebook)
REPO_ROOT = Path.cwd().parents[0]
DATA_PATH = REPO_ROOT / "data" / "raw" / "tetouan-power-consumption.csv"
CONFIG_PATH = REPO_ROOT / "project_config.yaml"

assert DATA_PATH.exists(), f"File not found: {DATA_PATH}"
config = ProjectConfig.from_yaml(str(CONFIG_PATH), env="dev")
pd.set_option("display.max_columns", 50)

## 2) Load Raw Data

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (52416, 9)


,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,1/1/2017 0:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
1,1/1/2017 0:10,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2,1/1/2017 0:20,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373


## 3) Column Mapping (Raw → snake_case)

Tetouan CSV has raw column names. We map them to snake_case for consistency.

| Raw Column | Renamed |
|------------|--------|
| DateTime | datetime |
| Temperature | temperature |
| Humidity | humidity |
| Wind Speed | wind_speed |
| general diffuse flows | general_diffuse_flows |
| diffuse flows | diffuse_flows |
| Zone 1 Power Consumption | zone1_consumption |
| Zone 2  Power Consumption | zone2_consumption |
| Zone 3  Power Consumption | zone3_consumption |

> Note: Zone 2 and Zone 3 have a double space in the raw name.

In [3]:
COLUMN_RENAME = {
    "DateTime": "datetime",
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "general diffuse flows": "general_diffuse_flows",
    "diffuse flows": "diffuse_flows",
    "Zone 1 Power Consumption": "zone1_consumption",
    "Zone 2  Power Consumption": "zone2_consumption",
    "Zone 3  Power Consumption": "zone3_consumption",
}

df = df.rename(columns=COLUMN_RENAME)
df.columns.tolist()

['datetime',
 'temperature',
 'humidity',
 'wind_speed',
 'general_diffuse_flows',
 'diffuse_flows',
 'zone1_consumption',
 'zone2_consumption',
 'zone3_consumption']

## 4) Parse DateTime

In [4]:
df["datetime"] = pd.to_datetime(df["datetime"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52416 entries, 0 to 52415
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   datetime               52416 non-null  datetime64[us]
 1   temperature            52416 non-null  float64       
 2   humidity               52416 non-null  float64       
 3   wind_speed             52416 non-null  float64       
 4   general_diffuse_flows  52416 non-null  float64       
 5   diffuse_flows          52416 non-null  float64       
 6   zone1_consumption      52416 non-null  float64       
 7   zone2_consumption      52416 non-null  float64       
 8   zone3_consumption      52416 non-null  float64       
dtypes: datetime64[us](1), float64(8)
memory usage: 3.6 MB


## 5) Temporal Features

Extract hour, day_of_week, month, is_weekend from datetime. These are **mandatory** for time series — EDA showed strong daily and seasonal patterns. They are in `project_config.yaml` `num_features`.

For time series, it is also done lags, but for this project im going to skip it.

In [5]:
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

## 6) Handle Missing Values

Tetouan dataset is mostly clean (no nulls in EDA). For robustness, we fill or drop if any appear.

In [6]:
null_counts = df.isnull().sum()
if null_counts.sum() > 0:
    print("Nulls found:", null_counts[null_counts > 0].to_dict())
    df = df.dropna()  # or fill with median/mean for numeric columns
else:
    print("No nulls found.")

No nulls found.


## 7) Select Columns & Generate id

Keep: `datetime` + `num_features` + `cat_features` + `[target]` + `["id"]`.  
**Keep `datetime`** — needed for time-based split and traceability.  
**`id`** — unique row identifier from datetime (for Delta tables, MLflow lineage). Use `datetime.astype(str)` so each row has a stable, interpretable id.

In [7]:
relevant_columns = ["datetime"] + config.num_features + config.cat_features + [config.target]
df = df[relevant_columns].copy()
df["id"] = df["datetime"].astype(str)

print("Final columns:", df.columns.tolist())
df.head(3)

Final columns: ['datetime', 'temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows', 'hour', 'day_of_week', 'month', 'is_weekend', 'zone1_consumption', 'id']


,datetime,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,hour,day_of_week,month,is_weekend,zone1_consumption,id
0,2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,0,6,1,1,34055.69620,2017-01-01 00:00:00
1,2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,0,6,1,1,29814.68354,2017-01-01 00:10:00
2,2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,0,6,1,1,29128.10127,2017-01-01 00:20:00


## 8) Sanity Check

Verifies preprocessing produced the expected output before moving on:
- **Columns** — exactly `datetime` + config columns + `id`
- **id dtype** — string (object), not numeric
- **No nulls** — all rows are complete

In [8]:
expected_columns = set(relevant_columns + ["id"])
assert set(df.columns) == expected_columns, f"Expected {expected_columns}, got {set(df.columns)}"
assert pd.api.types.is_string_dtype(df["id"]), "id must be string type"
assert df.isnull().sum().sum() == 0, "No nulls allowed"
print("✅ Preprocessing prototype complete.")

✅ Preprocessing prototype complete.


## 9) Time-Based Split

**No random split.** Tetouan is time series — we must preserve temporal order.  
From `docs/00-problem-statement-v1.md`:
- **Train:** 2017-01 → 2017-09
- **Val:** 2017-10 → 2017-11
- **Test:** 2017-12 → 2017-12-30

Random split would leak future data into training.

In [9]:
train_end = config.split.train_end if config.split else "2017-10-01"
val_end = config.split.val_end if config.split else "2017-12-01"

train_set = df[df["datetime"] < train_end]
val_set = df[(df["datetime"] >= train_end) & (df["datetime"] < val_end)]
test_set = df[df["datetime"] >= val_end]

print(f"Train: {len(train_set)} (Jan–Sep)")
print(f"Val:   {len(val_set)} (Oct–Nov)")
print(f"Test:  {len(test_set)} (Dec)")

print("\n")

print("Train Data")
print(f"- Min. Date: {train_set['datetime'].min()}")
print(f"- Max. Date: {train_set['datetime'].max()}")

print("\n")

print("Validation Data")
print(f"- Min. Date: {val_set['datetime'].min()}")
print(f"- Max. Date: {val_set['datetime'].max()}")

print("\n")

print("Test Data")
print(f"- Min. Date: {test_set['datetime'].min()}")
print(f"- Max. Date: {test_set['datetime'].max()}")


Train: 39312 (Jan–Sep)
Val:   8784 (Oct–Nov)
Test:  4320 (Dec)


Train Data
- Min. Date: 2017-01-01 00:00:00
- Max. Date: 2017-09-30 23:50:00


Validation Data
- Min. Date: 2017-10-01 00:00:00
- Max. Date: 2017-11-30 23:50:00


Test Data
- Min. Date: 2017-12-01 00:00:00
- Max. Date: 2017-12-30 23:50:00


---

**Next:** Extract this logic into `src/tetouan_power/data_processor.py` (Step 6b in Phase 1).  
`DataProcessor.split_data()` should use this time-based split, not random `train_test_split`.

# 10)  Generating synthetic data

In [11]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
rows = []
date_ranges = [
    # (start, end, count) — spread across the 3 split periods
    ("2017-01-15", "2017-09-15", 8),   # train period
    ("2017-10-10", "2017-11-20", 6),   # validation period
    ("2017-12-05", "2017-12-25", 6),   # test period
]

for start, end, n in date_ranges:
    timestamps = pd.date_range(start, end, periods=n)
    for ts in timestamps:
        rows.append({
            "DateTime": ts.strftime("%#m/%#d/%Y %H:%M"),  # Windows: %#m/%#d  Unix: %-m/%-d
            "Temperature": round(rng.uniform(5, 35), 3),
            "Humidity": round(rng.uniform(30, 90), 1),
            "Wind Speed": round(rng.uniform(0, 8), 3),
            "general diffuse flows": round(rng.uniform(0.01, 0.6), 3),
            "diffuse flows": round(rng.uniform(0.01, 0.5), 3),
            "Zone 1 Power Consumption": round(rng.uniform(20000, 50000), 4),
            "Zone 2  Power Consumption": round(rng.uniform(15000, 35000), 4),
            "Zone 3  Power Consumption": round(rng.uniform(18000, 40000), 4),
        })

df = pd.DataFrame(rows)

# Inject a few nulls so test_missing_value_handling exercises the dropna() path
df.loc[2, "Temperature"] = None
df.loc[5, "Humidity"] = None

df.to_csv("../tests/test_data/sample.csv", index=False)
print(f"Created {len(df)} rows ({df.isnull().sum().sum()} nulls injected)")
print(df.head(8))

Created 20 rows (2 nulls injected)
          DateTime  Temperature  Humidity  Wind Speed  general diffuse flows  \
0  1/15/2017 00:00       28.219      56.3       6.869                  0.421   
1  2/18/2017 17:08        8.843      57.0       2.966                  0.557   
2  3/25/2017 10:17          NaN      33.8       6.621                  0.383   
3  4/29/2017 03:25       28.352      41.7       3.734                  0.036   
4   6/2/2017 20:34       14.775      52.2       3.756                  0.122   
5   7/7/2017 13:42       18.115       NaN       5.602                  0.194   
6  8/11/2017 06:51       25.475      38.4       1.599                  0.014   
7  9/15/2017 00:00       18.767      64.1       1.118                  0.078   

   diffuse flows  Zone 1 Power Consumption  Zone 2  Power Consumption  \
0          0.056                49268.6705                 30222.7940   
1          0.325                44682.8484                 23868.2840   
2          0.381         